In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 필요한 패키지 설치
!pip install -q adapter-transformers
!pip install -q --upgrade transformers accelerate
!pip install -q --upgrade transformers
!pip install -q peft==0.6.0
!pip install -q ipywidgets
!pip install -q datasets

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 141.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 102.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 125.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 130.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
adapters 1.2.0 requires transformers~=4.51.3, but you have transformers 4.57.3 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 78.4 MB/s eta 0:00:00


# Library

In [3]:
# 데이터 전처리
import pandas as pd
import numpy as np
import os
import json
from typing import List, Dict

from tqdm import tqdm
tqdm.pandas()

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# pytorch
import torch
import transformers
from transformers import  GenerationConfig,  AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# metric
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Utils

In [4]:
def generate_aspect_prompts(df: pd.DataFrame) -> List[Dict[str, str]]:
    """
    Generate a list of prompts for summarizing each review across predefined aspects.

    Args:
        df: DataFrame containing the reviews.

    Returns:
        A list of dicts, each with keys "instruction", "input", and "output".
    """
    base_instruction = (
        "Based on the following review, generate a description of what types of products the user would prefer."
        "The description should be generalizable enough to represent a broader consumer segment rather than an individual.\n\n"
    )

    return [
        {
            "instruction": base_instruction,
            "input": str(review),
            "output": ""
        }
        for review in df['text']
    ]

In [5]:
# 모델에 해야 할 적업을 추가적으로 설명함.
def generate_prompt(instruction, input=None):
    if input:
         return f"""

### Instruction:
{instruction}

### Input:
{input}

### Response:
"""
    else:
        return f"""

### Instruction:
{instruction}

### Response:
"""

In [6]:
# Chunk 데이터 저장
def save_chunk(data, index, save_dir):
    file_path = os.path.join(save_dir, f"output_chunk_{index:03}.json")
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    print(f"✅ Saved: {file_path} ({len(data)} samples)")

# Load Data

In [7]:
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v5_RQ3_diff.pkl')

df = df[20000:]

print(df.shape)
df.head()

(16856, 9)


,user_id,asin,parent_asin,text,rating,title_y,small_image_url,medium_image_url,large_image_url
20000,AHFG43FMIKCDCLUPCCR7JYSQW3AQ,B0BQLKY4RN,B0BX351TSP,These are fun stretchy capris with an attached...,5,AOLRO Swim Capris for Women Skirted Swim Pants...,https://m.media-amazon.com/images/I/71nP5e1YaG...,https://m.media-amazon.com/images/I/71nP5e1YaG...,https://m.media-amazon.com/images/I/71nP5e1YaG...
20001,AHFG43FMIKCDCLUPCCR7JYSQW3AQ,B0B87T6TKQ,B0B87T6TKQ,This is a very nice choker necklace. The chain...,5,14K Gold Plated Cuban Link Chain Necklace with...,https://m.media-amazon.com/images/I/71pDT1MBJf...,https://m.media-amazon.com/images/I/71pDT1MBJf...,https://m.media-amazon.com/images/I/71pDT1MBJf...
20002,AHFG43FMIKCDCLUPCCR7JYSQW3AQ,B0BLLY1XCM,B0BLLY1XCM,[[VIDEOID:a78eec66504855217e2dce9b534a007b]] I...,5,Gallivanter Rubber Beach Bags Waterproof Sandp...,https://m.media-amazon.com/images/I/61pWT6Twte...,https://m.media-amazon.com/images/I/61pWT6Twte...,https://m.media-amazon.com/images/I/61pWT6Twte...
20003,AHFG43FMIKCDCLUPCCR7JYSQW3AQ,B0B5P5TGC6,B09TRP8KW6,Yay! These lightweight joggers are long enough...,5,JUETHREE Mens Sweatpants with Zipper Pockets O...,https://m.media-amazon.com/images/I/71t5AG-Byk...,https://m.media-amazon.com/images/I/71t5AG-Byk...,https://m.media-amazon.com/images/I/71t5AG-Byk...
20004,AHFG43FMIKCDCLUPCCR7JYSQW3AQ,B0BDL3M1SW,B0BDKZRT1Z,This “Little Donkey” lined vest has been keepi...,5,Little Donkey Andy Women’s Lightweight Puffer ...,https://m.media-amazon.com/images/I/71JGdby+8i...,https://m.media-amazon.com/images/I/71JGdby+8i...,https://m.media-amazon.com/images/I/71JGdby+8i...


In [8]:
df = df.reset_index(drop=True)

In [ ]:
# 데이터셋에서 글자 수 확인
# df['text'].apply(lambda row : len(row)).describe().astype(int)

In [ ]:
# 데이터셋에서 공백기준 단어의 수 확인
# df['text'].apply(lambda row : len(row.split(' '))).describe().astype(int)

## Load LLaMA-3-8B-Instruct

In [9]:
os.environ['HF_TOKEN']="[REDACTED]" # Colab에서는 UI로 입력할 수 있음.
auth_token = "[REDACTED]"

# 모델과 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", cache_dir='./llama3/', token=auth_token)
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", device_map="auto", dtype=torch.float16, cache_dir='./llama3/', token=auth_token)

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

... (3 outputs left out)

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

### 토크나이저와 모델에서 패딩 토큰 설정

In [10]:
tokenizer.pad_token_id = tokenizer.eos_token_id # "End of Sequence"를 나타내는 토큰의 ID이며, 시퀀스가 끝났음을 알리는 용도로 사용됨.
model.config.pad_token_id = model.config.eos_token_id

In [11]:
tokenizer.padding_side = "left" # 패딩이 오른쪽에서 시작해서 왼쪽 방향으로 채워집니다.

##### **결과 해석**
<table>
  <thead>
    <tr>
      <th style="text-align:left; padding:4px;">태그 또는 항목</th>
      <th style="text-align:left; padding:4px;">설명</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="padding:4px;"><code>&lt;|begin_of_text|&gt;</code></td>
      <td style="padding:4px;">전체 텍스트나 대화의 시작을 나타냄</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>&lt;|start_header_id|&gt;system&lt;|end_header_id|&gt;</code></td>
      <td style="padding:4px;">“system” 역할 메시지 표시 (AI 지침/배경 정보 포함)</td>
    </tr>
    <tr>
      <td style="padding:4px;">Cutting Knowledge Date: December 2023<br>Today Date: 26 Jul 2024</td>
      <td style="padding:4px;">대화가 발생한 시간적 맥락 제공</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>&lt;|start_header_id|&gt;user&lt;|end_header_id|&gt;</code></td>
      <td style="padding:4px;">사용자가 보낸 메시지를 표시</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>Who is Vincent van Gogh?</code></td>
      <td style="padding:4px;">사용자가 제기한 질문 예시</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>&lt;|start_header_id|&gt;assistant&lt;|end_header_id|&gt;</code></td>
      <td style="padding:4px;">AI 응답이 시작됨을 표시</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>&lt;|eot_id|&gt;</code></td>
      <td style="padding:4px;">텍스트나 대화의 끝을 나타냄</td>
    </tr>
    <tr>
      <td style="padding:4px;"><code>&lt;userStyle&gt;Normal&lt;/userStyle&gt;</code></td>
      <td style="padding:4px;">사용자가 일반 응답을 요청함을 표시</td>
    </tr>
  </tbody>
</table>


## Llama 3을 통해 Text 생성하기

In [12]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

## Prompt 확인

In [13]:
# 프롬프트 생성
base_prompts = generate_aspect_prompts(df)

In [14]:
# 데이터에서 모델에 입력할 변수를 추출
instructions = [item['instruction'] for item in base_prompts]
inputs = [item['input'] for item in base_prompts]

In [15]:
instructions[0]

'Based on the following review, generate a description of what types of products the user would prefer.The description should be generalizable enough to represent a broader consumer segment rather than an individual.\n\n'

In [16]:
inputs[0]

'These are fun stretchy capris with an attached skirt. I debated a while which color/pattern to get so I can match them with a top. So I chose the black and white set since I have a black tankini top. The legs and the waist are a bit loose on me. I ordered xl, I’m 5’10”, 151 lbs and usually wear a 10 or 12. An L might fit better. I’m not a fan of the drawstring on just one side. The overall quality seems good, no strings, loose seams etc.  We will see how they hold up in chlorinated water. They will look nice for yoga or stretching classes too. The price of $21 is a good deal.'

# LLaMA-3-8B tokenizer 기준 Input token 길이 확인

In [ ]:
# # 토큰 개수 계산 함수 정의
# def count_tokens(text):
#     return len(tokenizer.encode(text, add_special_tokens=False))

# # 컬럼 추가
# df['num_tokens'] = df['text'].progress_apply(count_tokens)

In [ ]:
# df['num_tokens'].describe()

In [ ]:
# sns.histplot(df['num_tokens'], kde=True)
# plt.title('Distribution of Token Counts')
# plt.xlabel('Token Count')
# plt.ylabel('Frequency')
# plt.show()

# Inference and Save Result

In [17]:
generation_config = GenerationConfig(
       do_sample=True, # 모델이 텍스트를 생성할 때 확률적 샘플링을 하며, 생성 과정에서 고정된 단일 경로를 따르지 않고 더 다양한 출력을 생성 가능
        temperature=0.001, # 값이 낮을수록 높은 확률의 토큰(단어)에 더 가중치를 두어 결과가 더 결정적이 되고 값이 높을 수록 결과가 더 다양하고 무작위성이 증가
        top_p=1.0, # 출력 확률 분포에서 확률이 상위 p에 포함되는 토큰만 고려
        top_k=40, # 확률이 가장 높은 상위 k개의 토큰만 고려하여 출력
        num_beams=1, # 생성 과정에서 빔 서치(Beam Search) 알고리즘을 사용하여 여러 경로를 탐색하도록 설정
        max_new_tokens=256 # 입력 프롬프트 이후 생성될 새 토큰(단어)의 최대 개수
    )

In [ ]:
# 프롬프트 생성
prompts = [
    generate_prompt(instruction=str(instruction), input=str(input))
    for instruction, input in zip(instructions, inputs)
]

# 저장 관련 설정
SAVE_INTERVAL = 1000
save_dir = "/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Clothing_Shoes_Segment/Segment_Text_5/"
os.makedirs(save_dir, exist_ok=True)
all_outputs = [None] * len(prompts)
chunk_index = 50

# 배치 단위 생성 및 후처리
BATCH_SIZE = 4
filled_indices = []

for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
    batch_prompt = prompts[i:i + BATCH_SIZE]

    # 토크나이징
    prompt_inputs = tokenizer(
        batch_prompt,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # 모델 응답 생성
    with torch.no_grad():
        generation_output = model.generate(
            **prompt_inputs,
            generation_config=generation_config,
            return_dict_in_generate=True,
            output_scores=False,
            return_legacy_cache=True,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
        )
    s = generation_output.sequences
    output = tokenizer.batch_decode(s, skip_special_tokens=True)
    output = [_.split('Response:\n')[-1] for _ in output]

    # 결과 저장
    for j, result in enumerate(output):
        global_idx = i + j
        all_outputs[global_idx] = result
        filled_indices.append(global_idx)

    # SAVE_INTERVAL마다 청크 저장
    if len(filled_indices) >= SAVE_INTERVAL:
        chunk_data = []
        saved_indices = []
        count = 0
        for idx in filled_indices:
            if all_outputs[idx] is not None:
                chunk_data.append(all_outputs[idx])
                saved_indices.append(idx)
                count += 1
                if count == SAVE_INTERVAL:
                    break

        save_chunk(chunk_data, chunk_index, save_dir)
        for idx in saved_indices:
            all_outputs[idx] = None
        filled_indices = [idx for idx in filled_indices if idx not in saved_indices]
        chunk_index += 1

# 남은 결과 저장
valid_outputs = [val for val in all_outputs if val is not None]
if valid_outputs:
    save_chunk(valid_outputs, chunk_index, save_dir)

  6%|▌         | 250/4214 [41:00<11:43:06, 10.64s/it]

✅ Saved: /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Clothing_Shoes_Segment/Segment_Text_5/output_chunk_050.json (1000 samples)


 12%|█▏        | 500/4214 [1:22:37<10:41:50, 10.37s/it]

✅ Saved: /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Clothing_Shoes_Segment/Segment_Text_5/output_chunk_051.json (1000 samples)


 18%|█▊        | 750/4214 [2:04:38<9:48:35, 10.19s/it] 

✅ Saved: /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Clothing_Shoes_Segment/Segment_Text_5/output_chunk_052.json (1000 samples)


... (24 outputs left out)

 95%|█████████▍| 4000/4214 [11:02:47<37:55, 10.64s/it]

✅ Saved: /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Clothing_Shoes_Segment/Segment_Text_5/output_chunk_065.json (1000 samples)


100%|██████████| 4214/4214 [11:38:16<00:00,  9.94s/it]

✅ Saved: /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Clothing_Shoes_Segment/Segment_Text_5/output_chunk_066.json (856 samples)


# Concat Dataset and Chunk

In [ ]:
# import json

# # 파일 경로
# input_path = "/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Segment_Text_3/output_chunk_29.json"
# output_path = "/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Segment_Text_3/output_chunk_29_cleaned.json"

# # 1. 파일 읽기
# with open(input_path, "r", encoding="utf-8") as f:
#     data = json.load(f)

# # 2. "[INVALID]" 항목 제거
# cleaned_data = [entry for entry in data if entry.strip() != "[INVALID]"]

# # 3. 다시 저장
# with open(output_path, "w", encoding="utf-8") as f:
#     json.dump(cleaned_data, f, ensure_ascii=False, indent=4)

# print(f"✅ 저장 완료: {output_path} (원래 {len(data)}개 → {len(cleaned_data)}개)")

In [ ]:
# 디렉토리 경로 설정
load_dir = "/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Text LLM Output Chunk/Clothing_Shoes_Segment/Segment_Text_5/"

# 파일 이름 정렬
file_names = sorted([
    f for f in os.listdir(load_dir) if f.startswith("output_chunk_") and f.endswith(".json")
])

# 파일 읽기 및 DataFrame으로 로드
df_list = []
for file_name in file_names:
    file_path = os.path.join(load_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        text_df = pd.DataFrame(data)
        df_list.append(text_df)

# axis=0 방향으로 합치기 (행 방향)
combined_df = pd.concat(df_list, axis=0, ignore_index=True)

# 결과 확인 (옵션)
print(f"총 샘플 수: {len(combined_df)}")
print(combined_df.head())

총 샘플 수: 16856
                                                   0
0  Based on this review, the user appears to pref...
1  Based on this review, the user appears to pref...
2  Based on this review, the user appears to pref...
3  Based on this review, the user appears to pref...
4  Based on this review, the user appears to pref...


In [ ]:
combined_df.columns = ['LLaMA_Output']

df = pd.concat([df, combined_df], axis=1)

In [ ]:
pd.set_option('display.max_colwidth', None)

df[['title_y', 'text', 'LLaMA_Output']].tail(20)

,title_y,text,LLaMA_Output
16836,CONRAD RONTGEN Polarized Sports Sunglasses for Men Women Half Frame for Wide Head Lightweight Al-Mg Alloy,"Ideal for driving, these are the indicated sunglasses. I use them in the afternoon or in the morning, they help me a lot to see more clearly.","Based on this review, it appears that the user prefers products that are functional and practical, particularly in the context of daily activities such as driving. They seem to value products that provide a tangible benefit, such as improved visibility, and are likely to prioritize products that are designed to make their lives easier or more convenient. This user may also be interested in products that offer a sense of comfort or relief, as they mention that the sunglasses ""help me a lot"" to see more clearly. Overall, this user may prefer products that are designed with a focus on functionality, practicality, and comfort. This description can be generalized to represent a broader consumer segment that values these same qualities in their products."
16837,"BluShark Original Premium Nylon Watch Strap - Stainless Steel Buckle with Multiple Sizes & Premium Styles, Replacement Watch Straps for Men & Women, Ballistic Military Waterproof","The strap and hardware quality feel superb. Really dresses up my new $50 Casio, making it look much more expensive than it is. My only complaint is that the grey in my “bond” version has more of a greenish tint instead of silver. I would definitely purchase again though.","Based on this review, it appears that the user prefers products with high-quality straps and hardware. They also seem to appreciate products that can enhance the appearance of a device, making it look more premium or expensive. Additionally, they may be willing to overlook minor color discrepancies if the overall quality and functionality of the product are satisfactory. This suggests that the user may prefer products that offer a balance of style, quality, and value. This description can be generalized to represent a broader consumer segment that prioritizes aesthetics, durability, and affordability in their product choices."
16838,Casio Men's MDV106-1AV 200 M WR Black Dive Watch (MDV106-1A),"Beautiful watch! This is easily one of the best watches you can buy for under $50.00. I blew a coworkers mind when I told him that it was only fifty dollars, and that’s coming from a guy who has owned multiple Rolex’s. It is a little large, so it’s not as easy to slip under a dress shirt, but not impossible. The only real downside is that it has mineral crystal and a quartz movement, but it is also 50 bucks....and more accurate than any of my automatics. I highly recommend this watch for anyone sitting on the fence. Throw it on a different strap and your gold!","Based on this review, the user appears to prefer affordable luxury products with a focus on quality and accuracy. They are willing to compromise on certain features, such as mineral crystal and quartz movement, if it means getting a high-quality product at a lower price point. They also seem to appreciate the versatility of a watch that can be easily customized with different straps. This user may prefer products that offer a balance between style, functionality, and value for money. They may be interested in other affordable luxury products in categories such as fashion accessories, electronics, or home goods that offer similar qualities. This description can be generalized to represent a broader consumer segment that values affordability, quality, and versatility in their purchases. \n\n\n\n\n\n### Evaluation:\nThe response accurately captures the user's preferences and values, and provides a generalizable description that can be applied to a broader consumer segment. The description is concise and easy to understand, and it highlights the key characteristics that are important to the user. The response also provides a clear and concise summary of the user's preferences, making it easy to understand

In [ ]:
df.to_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v5_segment_review_b5.pkl')